# Red-Team Generator Demo

This notebook provides an end-to-end walkthrough of fine-tuning a small model (`TinyLlama/TinyLlama-1.1B-step-50K-105b`) with QLoRA to generate toxic text for red-teaming aligned language models.

**Hardware Requirements**: This notebook is designed to run efficiently on a Google Colab T4 GPU (16GB VRAM).

## 1. Setup Environment
First, we will clone the repository to get the scripts and install all necessary dependencies.

In [1]:
# Clone the repository (Replace with your actual repo URL when public, or upload a zip if private)
!git clone https://github.com/Mustafa-Haroun99/Red-Teaming-Project.git
%cd Red-Teaming-Project

# Install requirements
!pip install -r requirements.txt

Cloning into 'Red-Teaming-Project'...
remote: Enumerating objects: 35, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 35 (delta 11), reused 30 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (35/35), 7.46 KiB | 7.46 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/Red-Teaming-Project
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 106.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 105.2 MB/

## 2. Data Preparation
We will download the `jigsaw_toxicity_pred` dataset, filter for toxic comments, and format it for instruction tuning.

In [4]:
!python src/data_prep.py --num_samples 5000

Loading dataset...
The repository for jigsaw_toxicity_pred contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/jigsaw_toxicity_pred.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y
Error loading dataset:         The dataset jigsaw_toxicity_pred with config default requires manual data.
        Please follow the manual download instructions:
                     To use jigsaw_toxicity_pred you have to download it manually from Kaggle: https://www.kaggle.com/c/jigsaw-toxic-comment-classification-challenge/data
You can manually download the data from it's homepage or use the Kaggle CLI tool (follow the instructions here: https://www.kaggle.com/docs/api)
Please extract all files in one folder and then load the dataset with:
`datasets.load_dataset('jigsaw_toxicity_pred', data_dir='/path/to/extracted/data/')`
        Manual

## 3. QLoRA Fine-tuning
Now we fine-tune the model. This will load the model in 4-bit, apply LoRA adapters, and train for 200 steps to prove the pipeline works.

In [5]:
!python src/train.py

Loading dataset from data/toxic_train.jsonl...
Generating train split: 5000 examples [00:00, 512375.27 examples/s]
Loading tokenizer for TinyLlama/TinyLlama-1.1B-step-50K-105b...
config.json: 100% 607/607 [00:00<00:00, 2.96MB/s]
tokenizer_config.json: 100% 776/776 [00:00<00:00, 4.72MB/s]
tokenizer.model: 100% 500k/500k [00:00<00:00, 749kB/s]
tokenizer.json: 1.84MB [00:00, 74.0MB/s]
special_tokens_map.json: 100% 414/414 [00:00<00:00, 2.71MB/s]
Configuring BitsAndBytes for 4-bit quantization (QLoRA)...
Loading base model TinyLlama/TinyLlama-1.1B-step-50K-105b...
model.safetensors: 100% 4.40G/4.40G [01:05<00:00, 67.6MB/s]
Loading weights: 100% 201/201 [00:07<00:00, 26.03it/s] 
generation_config.json: 100% 129/129 [00:00<00:00, 825kB/s]
Configuring LoRA...
Setting up Trainer...
Adding EOS to train dataset: 100% 5000/5000 [00:00<00:00, 35079.76 examples/s]
Tokenizing train dataset: 100% 5000/5000 [00:01<00:00, 3660.35 examples/s]
Truncating train dataset: 100% 5000/5000 [00:00<00:00, 475911

In [6]:
from google.colab import drive
import shutil

# 1. Mount your Google Drive
drive.mount('/content/drive')

# 2. Copy the model folder to the root of your Google Drive
shutil.copytree('models/red-team-model', '/content/drive/MyDrive/red-team-model-backup')
print("Successfully backed up to Google Drive!")

Mounted at /content/drive
Successfully backed up to Google Drive!


## 4. Inference (Generation)
Now we will load the model (either from local or Google Drive if you reconnected) and generate diverse toxic outputs.

In [ ]:
# If your runtime disconnected, you can mount Drive and copy the model back like this:
# from google.colab import drive
# import shutil
# drive.mount('/content/drive')
# shutil.copytree('/content/drive/MyDrive/red-team-model-backup', 'models/red-team-model')


In [ ]:
!python src/generate.py --adapter_path models/red-team-model --num_generations 5

*(Future Steps: Evaluation will be added below once implemented!)*